# Project 2B: Filter Assignment

This notebook defines the filter assignment workflow and reflection prompts.

Use these runnable notebooks:
1. [`Project_2b_DeadReckoning_RealReplay.ipynb`](Project_2b_DeadReckoning_RealReplay.ipynb)
2. [`Project_2b_FootholdEKF_RealReplay.ipynb`](Project_2b_FootholdEKF_RealReplay.ipynb)


## Dataset Setup

Download rosbags from:
- [Co-RaL GitHub page](https://github.com/SangwooJung98/Co-RaL-Dataset)
    - [Drive folder](https://drive.google.com/drive/folders/1rYxsiGvdnTWiiosqVWTdQBKY_Z0cAtJI?usp=sharing)

Place bags in `project_2b/data`:
1. `cd project_2b`
2. `mkdir -p data`
3. `cp /path/to/download/building.bag data/building.bag`
4. `cp /path/to/download/stair.bag data/stair.bag`

Start with `building.bag` first.


## 1) Run Dead Reckoning First (Building)

Run the dead-reckoning notebook on `building.bag`:
- `Project_2b_DeadReckoning_RealReplay.ipynb`

Deliverable:
1. Paste a screenshot of the main plots into `2c_Reflection_Questions.pptx`.
2. Add a short discussion of what is happening in dead reckoning (drift, attitude behavior, velocity behavior).


## 2) Prove the Dynamics Jacobian

The filter prediction propagates the body state by composing gravity, autonomous flow, and IMU increments.
\begin{equation}
X_{k+1}^- = W_k \, \phi(X_k^+) \, U_k.
\end{equation}


The transition Jacobian is:
\begin{equation}
A_k = \operatorname{Ad}_{U_k^{-1}} \, \Phi_k,
\end{equation}

with
\begin{equation}
\Phi_k = \begin{bmatrix}
I_3 & 0 & 0 \\
0 & I_3 & \Delta t\,I_3 \\
0 & 0 & I_3
\end{bmatrix}.
\end{equation}

Recall that $\phi(X)$ is the autonomous flow. It integrates the NavState forward by $\Delta t$ under gravity alone (zero IMU input).

For the augmented body+foothold state, the transition matrix is block diagonal:
\begin{equation}
F_k = \operatorname{blkdiag}(A_k, I_{3N}).
\end{equation}

Your task is to prove the Jacobian expression above from first-order linearization. You can use the Jacobians from compose as we did in the first assignment. 

Deliverable:
1. Submit your derivation in the corresponding slide in `2c_Reflection_Questions.pptx`.


## 3) Measurement Jacobian: Given Equation, Prove It

The contact measurement maps body pose and one foothold landmark into body coordinates.
\begin{equation}
h_i(x) = R^T(f_i - p).
\end{equation}

The innovation is built from measurement minus prediction.
\begin{equation}
r_i = z_i - h_i(\hat x^-).
\end{equation}

Given answer (Jacobian of $h_i$ w.r.t. the error state ordering $[\delta\phi,\delta p,\delta v,\delta f_1,\ldots,\delta f_N]$):
\begin{equation}
H_i = \begin{bmatrix}
-[\hat z_i]_\times & -R^T & 0_{3\times 3} & 0 & \cdots & R^T & \cdots & 0
\end{bmatrix},
\end{equation}

where the $R^T$ foothold block is at foot $i$, and all other foothold blocks are zero.

Here $\hat{z}_i = R^T(\hat{f}_i - \hat{p})$ is the predicted contact vector in body frame evaluated at the prior estimate.

Your task is to prove this Jacobian and clearly justify the zero-block structure.

Deliverable:
1. Submit your derivation in the corresponding slide in `2c_Reflection_Questions.pptx`.


## 4) Run Foothold Filter (Building)

*Note: for Tasks 4–5, leave APPLY_HEIGHT_PRIOR = False. You will toggle it in Task 7.*

Run the foothold-EKF notebook on `building.bag`:
- `Project_2b_FootholdEKF_RealReplay.ipynb`

Deliverable:
1. Paste screenshots into `2c_Reflection_Questions.pptx`.
2. Compare against dead reckoning and discuss what changed.


## 5) Reflection: Yaw Behavior

In the reflection PowerPoint, answer:
- What is happening with yaw over time?
- Why can yaw still drift or be weakly observable in this setup?


## 6) Reflection: Additional Sensors

In the reflection PowerPoint, propose how you would improve the system with other sensors.

Prompt:
- Which sensor(s) would you add (e.g., magnetometer, camera, LiDAR, GPS, wheel odometry, radar constraints)?
- Where would each enter the filter (measurement model/state augmentation)?
- What failure mode would each sensor help with?


## 7) Stair Dataset + Height Prior Ablation

Now run `stair.bag` in the foothold notebook and compare two runs:
1. `APPLY_HEIGHT_PRIOR = True`
2. `APPLY_HEIGHT_PRIOR = False`

Deliverable:
1. Paste screenshots from both runs into `2c_Reflection_Questions.pptx`.
2. Write observations on what changes with and without the height prior.
